# file:h5msm

H5MSM is MolSysMT's versioned native persistence format for topology, chemical states, and structural arrays. New files use schema 0.4. MolSysMT continues to read 0.3 files and migrates their legacy bonds and components into one reference chemical state. Unknown future versions are rejected rather than guessed.

Schema 0.4 stores stable atom and hierarchy data, including nullable isotope mass numbers, separately from `/topology/chemical_states`. Each state owns its components, atom-to-component membership, optional atom chemistry, complete normalized rich bond table, completeness flags, and optional provenance reference. `/structures/chemical_state_index` stores the nullable per-frame association owned by `MolSys`; `-1` means unknown, and the global reference is never written as if it were frame-specific evidence. Missing columns, nullable values, explicit zero, and `False` remain distinct. When a reference state exists, legacy HDF5 paths are compatibility hard links to the same datasets rather than duplicate storage. Public `get()` and `has_attribute()` can read the persisted isotope, atom chemistry, and rich bond fields without requiring direct HDF5 access.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import molsysmt as msm
from molsysmt.native import Topology

topology = Topology(n_atoms=3)
msm.set(topology, element='atom', isotope=[13, None, 2])
msm.set(topology, element='atom', formal_charge=[0, -1, 1])

with TemporaryDirectory() as directory:
    filename = Path(directory) / 'system.h5msm'
    msm.convert(topology, to_form='file:h5msm', output_filename=filename)
    restored = msm.convert(filename, to_form='molsysmt.Topology')
    state_info = msm.get(
        restored, element='system', n_chemical_states=True,
        reference_chemical_state_index=True,
    )

state_info

[1, 0]